# 07. 문서 로딩 및 임베딩 추적

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. TextLoader로 문서를 로딩할 때 LangSmith Trace가 어떻게 기록되는지 확인할 수 있어요
2. RecursiveCharacterTextSplitter의 청킹 파라미터가 Trace에 어떻게 나타나는지 분석할 수 있어요
3. OpenAIEmbeddings 호출이 LangSmith에서 얼마나 많은 API 요청을 발생시키는지 파악할 수 있어요
4. FAISS VectorStore의 저장과 조회 과정을 Trace로 추적할 수 있어요
5. `@traceable(run_type="retriever")`를 사용하여 커스텀 retriever 함수를 추적할 수 있어요

## 사전 지식

- 06_RAG 과정에서 배운 8단계 RAG 파이프라인
- `TextLoader`, `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `FAISS` 기본 사용법
- LangSmith 환경 설정 (01-Setup-and-Overview 완료)
- `@traceable` 기본 사용법 (02-First-Trace 완료)

## RAG 전처리 단계와 LangSmith 추적

RAG 파이프라인은 크게 두 단계로 나뉘어요.

- **전처리(Pre-processing)**: 문서 로딩 → 청킹 → 임베딩 → 벡터 저장
- **실행(Runtime)**: 사용자 질문 → 검색 → 프롬프트 구성 → 생성

이 노트북에서는 전처리 단계 각 단계에서 LangSmith가 무엇을 기록하는지 살펴볼게요.

```mermaid
flowchart TD
    A["📄 문서 파일<br/>(ai_concepts.txt)"] --> B["TextLoader<br/>로딩"]
    B --> C["RecursiveCharacterTextSplitter<br/>청킹"]
    C --> D["OpenAIEmbeddings<br/>임베딩 생성"]
    D --> E["FAISS VectorStore<br/>저장"]
    E --> F["similarity_search()<br/>조회"]
    F --> G["@traceable retriever<br/>커스텀 추적"]

    B --> LS1["🔍 LangSmith<br/>Trace 기록"]
    D --> LS2["🔍 API 호출<br/>토큰/비용"]
    F --> LS3["🔍 검색 결과<br/>문서 목록"]

    classDef doc fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef langsmith fill:#fff3cd,stroke:#ffc107,color:#856404

    class A doc
    class B,C,G process
    class D,E,F storage
    class LS1,LS2,LS3 langsmith
```

### RAG 전처리 단계별 LangSmith 추적 포인트

| 단계 | LangSmith가 기록하는 것 | 주목할 점 |
|------|------------------------|----------|
| 문서 로딩 | 파일 경로, 로딩 시간 | 파일 크기에 따른 지연 시간 |
| 청킹 | 원본 문서 수, 청크 수, 파라미터 | chunk_size, overlap 설정 영향 |
| 임베딩 | API 호출 횟수, 배치 크기, 비용 | 청크 수 = API 배치 호출 수 |
| VectorStore | 저장 시간, 인덱스 크기 | FAISS는 로컬 인메모리 |
| 검색 | 쿼리, 반환 문서, 유사도 점수 | run_type="retriever" 특별 UI |

## 환경 설정

In [10]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # FAISS + OpenMP 충돌 방지
from dotenv import load_dotenv

load_dotenv()
print("환경 변수가 로드되었어요.")

환경 변수가 로드되었어요.


In [11]:
# ---------------------------------------------------
# LangSmith 프로젝트 설정
# ---------------------------------------------------
# 이 노트북의 Trace를 별도 프로젝트로 관리해요
import os

os.environ["LANGSMITH_PROJECT"] = "Ch07-Document-Loading"  # 프로젝트 이름

print(f"프로젝트: {os.environ.get('LANGSMITH_PROJECT')}")

프로젝트: Ch07-Document-Loading


## 1. 문서 로딩 추적 (TextLoader)

TextLoader로 파일을 로딩하면 LangSmith에 Document 로딩 Trace가 기록돼요.

> 🔑 **핵심 개념**: LangChain의 모든 컴포넌트는 `LANGCHAIN_TRACING_V2=true`만 설정하면 자동으로 추적돼요. 별도 코드 수정 없이 TextLoader, TextSplitter, Embeddings, Retriever 모두 자동 추적됩니다.

> 🎯 **강의 포인트**: TextLoader.load()를 실행한 후 LangSmith UI를 열어보세요. `rag-document-loading` 프로젝트에 새로운 Trace가 생성되어 있어요. 클릭하면 파일 경로, 로딩 시간, 반환된 Document 수를 확인할 수 있어요.

In [12]:
# ---------------------------------------------------
# TextLoader: 텍스트 파일 로딩
# ---------------------------------------------------
# TextLoader는 일반 텍스트 파일을 Document 객체로 변환해요
# LangSmith에 자동으로 Trace가 기록돼요
from langchain_community.document_loaders import TextLoader

# 샘플 데이터 파일 경로
file_path = "data/ai_concepts.txt"

# TextLoader로 파일 로딩
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

# 로딩 결과 확인
print(f"로딩된 Document 수: {len(documents)}")
print(f"첫 번째 Document 메타데이터: {documents[0].metadata}")
print(f"첫 번째 Document 내용 (앞 200자):")
print(documents[0].page_content[:200])

로딩된 Document 수: 1
첫 번째 Document 메타데이터: {'source': 'data/ai_concepts.txt'}
첫 번째 Document 내용 (앞 200자):
# 인공지능과 머신러닝 기초 개념

## 1. 머신러닝이란?

머신러닝(Machine Learning)은 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 학습하는 인공지능의 한 분야입니다.
전통적인 프로그래밍에서는 개발자가 규칙을 직접 작성하지만, 머신러닝에서는 컴퓨터가 데이터에서 패턴을 스스로 발견합니다.
머신러닝의 주요 유형으로는 지도 학습, 비지도 학습


In [13]:
# ---------------------------------------------------
# LangSmith 추적 완료 대기
# ---------------------------------------------------
# 비동기 추적이 모두 제출될 때까지 기다려요
from langchain_core.tracers.langchain import wait_for_all_tracers

wait_for_all_tracers()
print("Trace가 LangSmith에 제출되었어요.")
print("👉 LangSmith UI에서 'rag-document-loading' 프로젝트를 확인해보세요.")

Trace가 LangSmith에 제출되었어요.
👉 LangSmith UI에서 'rag-document-loading' 프로젝트를 확인해보세요.


## 2. 텍스트 청킹 추적 (RecursiveCharacterTextSplitter)

청킹(Chunking)은 긴 문서를 모델이 처리 가능한 크기로 나누는 과정이에요.
이 단계의 파라미터 설정이 검색 품질에 큰 영향을 미쳐요.

> 💡 **실무 팁**: `chunk_size=1000, chunk_overlap=100`은 일반적인 출발점이에요. 너무 작은 청크(200 이하)는 맥락이 부족하고, 너무 큰 청크(2000 이상)는 임베딩 비용이 증가해요. 도메인과 문서 특성에 맞게 조정해야 해요.

> ⚠️ **자주 하는 실수**: `chunk_overlap`을 0으로 설정하면 청크 경계에서 문장이 잘려 검색 품질이 떨어질 수 있어요. 일반적으로 `chunk_size`의 10~20% 수준으로 설정해요.

In [14]:
# ---------------------------------------------------
# RecursiveCharacterTextSplitter: 텍스트 청킹
# ---------------------------------------------------
# 재귀적 방법으로 문단 → 문장 → 단어 순으로 분할해요
# split_documents() 호출도 LangSmith에 자동 추적돼요
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # 각 청크의 최대 문자 수
    chunk_overlap=50,     # 청크 간 겹치는 문자 수 (맥락 유지)
    length_function=len,  # 길이 측정 함수
)

# 문서 분할
split_docs = text_splitter.split_documents(documents)

# 분할 결과 확인
print(f"원본 Document 수: {len(documents)}")
print(f"분할 후 청크 수: {len(split_docs)}")
print(f"\n첫 번째 청크:")
print(f"  내용 (앞 150자): {split_docs[0].page_content[:150]}")
print(f"  길이: {len(split_docs[0].page_content)}자")
print(f"\n두 번째 청크:")
print(f"  내용 (앞 150자): {split_docs[1].page_content[:150]}")
print(f"  길이: {len(split_docs[1].page_content)}자")

원본 Document 수: 1
분할 후 청크 수: 6

첫 번째 청크:
  내용 (앞 150자): # 인공지능과 머신러닝 기초 개념

## 1. 머신러닝이란?

머신러닝(Machine Learning)은 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 학습하는 인공지능의 한 분야입니다.
전통적인 프로그래밍에서는 개발자가 규칙을 직접 작성하지만, 머신러닝에서는 컴퓨터가
  길이: 425자

두 번째 청크:
  내용 (앞 150자): ## 3. 자연어 처리(NLP)

자연어 처리(Natural Language Processing, NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.
주요 NLP 작업으로는 텍스트 분류, 감성 분석, 기계 번역, 질문 응답, 텍스트 요약 등이 있습니다.
최
  길이: 489자


In [15]:
# ---------------------------------------------------
# 청크 크기 분포 분석
# ---------------------------------------------------
# 어떤 크기의 청크들이 생성되었는지 확인해요
chunk_sizes = [len(doc.page_content) for doc in split_docs]

print(f"청크 크기 통계:")
print(f"  최소: {min(chunk_sizes)}자")
print(f"  최대: {max(chunk_sizes)}자")
print(f"  평균: {sum(chunk_sizes) / len(chunk_sizes):.1f}자")
print(f"\n각 청크의 처음 50자:")
for i, doc in enumerate(split_docs[:5]):
    print(f"  청크 {i+1} ({len(doc.page_content)}자): {doc.page_content[:50]}...")

청크 크기 통계:
  최소: 373자
  최대: 489자
  평균: 425.2자

각 청크의 처음 50자:
  청크 1 (425자): # 인공지능과 머신러닝 기초 개념

## 1. 머신러닝이란?

머신러닝(Machine Le...
  청크 2 (489자): ## 3. 자연어 처리(NLP)

자연어 처리(Natural Language Process...
  청크 3 (465자): ## 5. RAG(Retrieval-Augmented Generation)

RAG는 외부...
  청크 4 (399자): ## 7. 임베딩(Embedding)

임베딩은 텍스트, 이미지 등의 데이터를 고차원 숫자...
  청크 5 (373자): ## 9. 파인튜닝(Fine-tuning)

파인튜닝은 사전 학습된 모델을 특정 작업이나 ...


In [16]:
# ============================================================
# TODO: chunk_size와 chunk_overlap을 변경해서 청크 수 변화를 관찰해요
# 힌트: chunk_size를 200, 500, 1000으로 바꿔서 청크 수를 비교해요
# 예상 결과: chunk_size가 클수록 청크 수가 줄어들고, 각 청크가 더 길어져요
# ============================================================

# 비교 실험: 다른 chunk_size로 분할
splitter_small = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
splitter_large = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

small_chunks = splitter_small.split_documents(documents)
large_chunks = splitter_large.split_documents(documents)

print("chunk_size 비교 실험:")
print(f"  chunk_size=200: {len(small_chunks)}개 청크")
print(f"  chunk_size=500: {len(split_docs)}개 청크 (현재)")
print(f"  chunk_size=1000: {len(large_chunks)}개 청크")
print("\n💡 LangSmith UI에서 각 split_documents() 호출의 Trace를 비교해보세요!")

wait_for_all_tracers()

chunk_size 비교 실험:
  chunk_size=200: 20개 청크
  chunk_size=500: 6개 청크 (현재)
  chunk_size=1000: 3개 청크

💡 LangSmith UI에서 각 split_documents() 호출의 Trace를 비교해보세요!


## 3. 임베딩 생성 추적 (OpenAIEmbeddings)

임베딩 생성 단계에서 LangSmith는 OpenAI API 호출 내역을 상세하게 기록해요.
특히 **토큰 사용량**과 **API 호출 횟수**를 추적하여 비용 관리에 활용할 수 있어요.

> 🔑 **핵심 개념**: `FAISS.from_documents()`를 호출하면 내부적으로 청크 수에 따라 여러 번의 OpenAI Embeddings API를 배치(batch)로 호출해요. LangSmith는 이 모든 API 호출을 개별 Span으로 기록하여 총 비용을 계산해줘요.

> 💡 **실무 팁**: `text-embedding-3-small`은 `text-embedding-3-large`보다 가격이 5배 저렴하면서 성능 차이가 크지 않아요. 비용 최적화를 위해 먼저 small 모델로 시작하는 것을 권장해요.

In [ ]:
# ---------------------------------------------------
# OpenAIEmbeddings: 임베딩 모델 초기화
# ---------------------------------------------------
# text-embedding-3-small: 비용 효율적, 1536차원 벡터
# text-embedding-3-large: 더 높은 품질, 3072차원 벡터
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 비용 효율적인 임베딩 모델
)

print("임베딩 모델 초기화 완료:")
print(f"  모델: text-embedding-3-small")
print(f"  예상 차원: 1536")

In [ ]:
# ---------------------------------------------------
# FAISS VectorStore 생성 (임베딩 포함)
# ---------------------------------------------------
# from_documents()는 내부적으로:
#   1. 각 청크를 텍스트로 추출
#   2. OpenAI Embeddings API 호출 (배치 처리)
#   3. FAISS 인덱스에 벡터 저장
# 이 모든 과정이 LangSmith에 Trace로 기록돼요
from langchain_community.vectorstores import FAISS

print(f"임베딩 생성 중... ({len(split_docs)}개 청크)")
print("LangSmith UI에서 토큰 사용량을 확인할 수 있어요.")

# 임베딩 + FAISS 인덱스 생성 (LangSmith 자동 추적)
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

print(f"\nVectorStore 생성 완료!")
print(f"  저장된 벡터 수: {vectorstore.index.ntotal}")

In [ ]:
# ---------------------------------------------------
# 단일 텍스트 임베딩 테스트
# ---------------------------------------------------
# embed_query()로 단일 텍스트의 임베딩 벡터를 확인해요
# 이 호출도 LangSmith에 Trace가 기록돼요
sample_text = "RAG란 무엇인가요?"
embedding_vector = embeddings.embed_query(sample_text)

print(f"임베딩 테스트:")
print(f"  입력 텍스트: {sample_text}")
print(f"  벡터 차원: {len(embedding_vector)}")
print(f"  벡터 앞 5개 값: {[f'{v:.4f}' for v in embedding_vector[:5]]}")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 EmbeddingsOpenAI Trace의 토큰 사용량을 확인해보세요!")

## 4. VectorStore 조회 추적 (similarity_search)

VectorStore에서 유사도 검색을 수행하면 LangSmith가 **쿼리**, **반환 문서**, **검색 시간**을 기록해요.

> 🎯 **강의 포인트**: `similarity_search()`를 실행한 후 LangSmith UI를 보면, VectorStoreRetriever run type의 Trace에서 검색된 문서 목록과 각 문서의 메타데이터를 확인할 수 있어요. 이것이 바로 "검색이 제대로 됐는가"를 디버깅하는 첫 번째 단계예요.

In [ ]:
# ---------------------------------------------------
# similarity_search: 유사도 기반 검색
# ---------------------------------------------------
# 쿼리 텍스트와 가장 유사한 k개의 청크를 반환해요
# as_retriever()를 통한 호출도 LangSmith에 추적돼요
retriever = vectorstore.as_retriever(
    search_type="similarity",  # 코사인 유사도 검색
    search_kwargs={"k": 3}     # 상위 3개 결과 반환
)

# 검색 실행
query = "RAG 파이프라인의 단계는 무엇인가요?"
retrieved_docs = retriever.invoke(query)

print(f"검색 쿼리: {query}")
print(f"반환된 문서 수: {len(retrieved_docs)}")
print("\n검색된 문서:")
for i, doc in enumerate(retrieved_docs):
    print(f"\n  [{i+1}] (소스: {doc.metadata.get('source', '알 수 없음')})")
    print(f"  내용: {doc.page_content[:200]}...")

In [ ]:
# ---------------------------------------------------
# 여러 쿼리로 검색 테스트
# ---------------------------------------------------
# 다양한 쿼리로 검색하여 Trace를 LangSmith에 쌓아요
test_queries = [
    "머신러닝과 딥러닝의 차이",
    "벡터 데이터베이스는 무엇인가요?",
    "LangSmith로 무엇을 할 수 있나요?",
    "프롬프트 엔지니어링 기법",
]

print("여러 쿼리 검색 실행:")
for q in test_queries:
    docs = retriever.invoke(q)
    print(f"  '{q}' → {len(docs)}개 문서 반환")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 Retriever run type의 Trace들을 확인해보세요!")
print("   각 Trace의 Inputs/Outputs를 보면 쿼리와 반환 문서를 확인할 수 있어요.")

## 5. 커스텀 Retriever 추적 (@traceable)

기본 LangChain 컴포넌트는 자동으로 추적되지만, **자신이 작성한 함수**는 `@traceable`로 명시적으로 추적해야 해요.
특히 `run_type="retriever"`를 지정하면 LangSmith UI에서 **검색 결과 특화 뷰**로 표시돼요.

> 🔑 **핵심 개념**: `run_type="retriever"`를 지정한 함수는 반환값이 `{"page_content": str, "type": "Document", "metadata": dict}` 형태의 딕셔너리 리스트여야 해요. 이 형식을 따라야 LangSmith UI에서 문서별 `page_content`와 메타데이터를 구조적으로 표시해줘요.

> ⚠️ **자주 하는 실수**: `run_type="retriever"`를 지정했는데 Document 객체를 그대로 반환하면 UI가 올바르게 표시되지 않아요. 딕셔너리 형태로 변환해서 반환해야 해요.

In [ ]:
# ---------------------------------------------------
# @traceable(run_type="retriever"): 커스텀 Retriever 함수
# ---------------------------------------------------
# run_type="retriever"를 지정하면 LangSmith에서
# 검색 결과 특화 UI로 표시돼요 (page_content, metadata 구조적 표시)
from langsmith import traceable

@traceable(
    run_type="retriever",          # LangSmith 검색 결과 특화 뷰
    name="custom_rag_retriever",   # Trace에 표시될 이름
    tags=["custom", "faiss"],      # 분류용 태그
)
def custom_retriever(query: str, k: int = 3) -> list:
    """커스텀 retriever: 유사도 검색 + 결과 필터링"""
    # FAISS에서 검색 (기본 k보다 더 많이 가져와서 필터링)
    raw_docs = vectorstore.similarity_search(query, k=k * 2)

    # 특정 키워드가 포함된 문서만 필터링 (예시)
    filtered = [doc for doc in raw_docs if len(doc.page_content) > 100]

    # LangSmith retriever 형식으로 변환
    # 중요: 딕셔너리 리스트로 반환해야 해요
    return [
        {
            "page_content": doc.page_content,
            "type": "Document",
            "metadata": {
                **doc.metadata,
                "rank": i + 1,          # 검색 순위
                "char_count": len(doc.page_content),  # 문자 수
            }
        }
        for i, doc in enumerate(filtered[:k])
    ]

In [ ]:
# ---------------------------------------------------
# 커스텀 Retriever 실행 및 결과 확인
# ---------------------------------------------------
# custom_retriever() 호출 → LangSmith에 Trace 기록
query = "임베딩과 벡터 데이터베이스 관계"
results = custom_retriever(query, k=3)

print(f"커스텀 Retriever 결과:")
print(f"  쿼리: {query}")
print(f"  반환된 문서 수: {len(results)}")

for i, doc in enumerate(results):
    print(f"\n  [{doc['metadata']['rank']}] 순위 문서")
    print(f"  내용: {doc['page_content'][:150]}...")
    print(f"  메타데이터: {doc['metadata']}")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 'custom_rag_retriever' Trace를 찾아보세요!")
print("   run_type=retriever이므로 Documents 탭에서 결과를 구조적으로 볼 수 있어요.")

In [ ]:
# ---------------------------------------------------
# @traceable로 전처리 전체를 하나의 Trace로 묶기
# ---------------------------------------------------
# 문서 로딩부터 VectorStore 생성까지 하나의 부모 Trace로 감싸요
@traceable(name="rag_preprocessing", tags=["preprocessing", "setup"])
def build_vectorstore_from_file(file_path: str, chunk_size: int = 500) -> dict:
    """RAG 전처리 파이프라인: 파일 → VectorStore"""
    # Step 1: 문서 로딩
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()

    # Step 2: 청킹
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_size // 10,
    )
    chunks = splitter.split_documents(docs)

    # Step 3: 임베딩 + VectorStore 생성
    vs = FAISS.from_documents(documents=chunks, embedding=embeddings)

    return {
        "num_docs": len(docs),
        "num_chunks": len(chunks),
        "chunk_size": chunk_size,
        "vector_count": vs.index.ntotal,
    }

# 전처리 실행 (전체가 하나의 부모 Trace 아래 묶여요)
result = build_vectorstore_from_file("data/ai_concepts.txt", chunk_size=500)

print("전처리 결과:")
for key, value in result.items():
    print(f"  {key}: {value}")

wait_for_all_tracers()
print("\n👉 LangSmith에서 'rag_preprocessing' Trace를 보면")
print("   TextLoader → TextSplitter → Embeddings 순으로 계층 구조가 보여요!")

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **자동 추적**: `LANGCHAIN_TRACING_V2=true` 설정만으로 TextLoader, TextSplitter, Embeddings, VectorStore 모두 자동으로 LangSmith에 기록돼요
- **임베딩 비용 추적**: `FAISS.from_documents()`가 내부적으로 얼마나 많은 Embeddings API를 호출하는지 LangSmith에서 확인할 수 있어요
- **run_type="retriever"**: `@traceable(run_type="retriever")`를 사용하면 LangSmith UI에서 검색 결과를 구조적으로 표시해줘요
- **부모-자식 Trace**: `@traceable`로 감싼 함수 내부의 LangChain 컴포넌트는 자동으로 자식 Span이 되어 계층 구조를 형성해요
- **반환 형식**: `run_type="retriever"` 함수는 `{"page_content": str, "type": "Document", "metadata": dict}` 딕셔너리 리스트를 반환해야 해요

## 다음 노트북 예고

다음 `08-Retrieval-Analysis.ipynb`에서는 **검색 품질 분석**을 배워요. similarity vs MMR vs EnsembleRetriever의 검색 결과를 LangSmith에서 비교하고, Reranker 적용 전후의 Trace를 분석해요.